# Random Graph Analysis using PageRank, PCA, and GMM Clustering

### Authors
Sudarshan Damodharan, Adithya Bhaskara, Ali Haroon

## Purpose

This notebook compares three random graph models:

- **Erdős–Rényi (ER)**
- **Watts–Strogatz (WS)**
- **Barabási–Albert (BA)**

The main questions are:

1. How does graph structure affect **influence concentration**, measured using **PageRank**?
2. Can graph-derived node features reveal distinct **node roles** through **PCA** and **GMM clustering**?
3. Does **targeted antidote allocation** outperform random allocation in a simulated spread experiment?

The notebook is written so that you can rerun it end-to-end and export report-ready figures directly into a `figures/` folder for use in LaTeX.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

In [ ]:
# Global plotting style for report-quality figures
plt.rcParams.update({
    "figure.figsize": (7.0, 4.5),
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "lines.linewidth": 2,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

pd.set_option("display.max_columns", 50)
pd.set_option("display.precision", 4)

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

MODEL_ORDER = ["ER", "WS", "BA"]
STRATEGY_ORDER = ["none", "targeted_pagerank", "random"]

MODEL_LABELS = {
    "ER": "Erdős–Rényi",
    "WS": "Watts–Strogatz",
    "BA": "Barabási–Albert",
}

STRATEGY_LABELS = {
    "none": "No intervention",
    "targeted_pagerank": "Top-5 PageRank removed",
    "random": "5 random nodes removed",
}

def save_figure(name: str, tight: bool = True) -> None:
    if tight:
        plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")

## Reproducibility and Runtime

The defaults below are chosen so that the notebook is substantial but still practical to rerun on a laptop.

- `N_ANALYSIS = 300` keeps repeated centrality calculations manageable.
- `NUM_GRAPH_SEEDS = 8` gives repeated experiments across graph realizations.
- `SPREAD_REPS = 15` gives repeated spread simulations per graph and intervention strategy.

You can raise these later if you want tighter estimates.

In [ ]:
# Core analysis parameters
RANDOM_STATE = 42

N_ANALYSIS = 300
N_GRAPH_PLOT = 120   # smaller graph for cleaner network visualization

NUM_GRAPH_SEEDS = 8
SPREAD_STEPS = 20
SPREAD_BETA = 0.20
SPREAD_REPS = 15
K_REMOVE = 5

EXAMPLE_SEED = 0

def get_default_graph_params(n: int) -> dict[str, dict]:
    # Roughly matched expected average degree across models
    return {
        "ER": {"p": 6 / (n - 1)},
        "WS": {"k": 6, "beta": 0.20},
        "BA": {"m": 3},
    }

ANALYSIS_PARAMS = get_default_graph_params(N_ANALYSIS)
PLOT_PARAMS = get_default_graph_params(N_GRAPH_PLOT)

## 1. Graph Generation

In [ ]:
def generate_erdos_renyi(n: int, p: float, seed: int | None = None) -> nx.Graph:
    return nx.erdos_renyi_graph(n=n, p=p, seed=seed)

def generate_watts_strogatz(n: int, k: int, beta: float, seed: int | None = None) -> nx.Graph:
    return nx.watts_strogatz_graph(n=n, k=k, p=beta, seed=seed)

def generate_barabasi_albert(n: int, m: int, seed: int | None = None) -> nx.Graph:
    return nx.barabasi_albert_graph(n=n, m=m, seed=seed)

def generate_graph(model: str, n: int, seed: int | None = None, **kwargs) -> nx.Graph:
    model = model.upper()
    if model == "ER":
        return generate_erdos_renyi(n=n, p=kwargs["p"], seed=seed)
    if model == "WS":
        return generate_watts_strogatz(n=n, k=kwargs["k"], beta=kwargs["beta"], seed=seed)
    if model == "BA":
        return generate_barabasi_albert(n=n, m=kwargs["m"], seed=seed)
    raise ValueError(f"Unknown model: {model}")

In [ ]:
def plot_example_graphs(n: int = N_GRAPH_PLOT, seed: int = EXAMPLE_SEED, filename: str = "random_graph_examples") -> None:
    params = get_default_graph_params(n)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

    for ax, model in zip(axes, MODEL_ORDER):
        G = generate_graph(model, n=n, seed=seed, **params[model])
        pos = nx.spring_layout(G, seed=seed)
        nx.draw_networkx_nodes(G, pos, node_size=16, ax=ax, alpha=0.9)
        nx.draw_networkx_edges(G, pos, width=0.4, alpha=0.25, ax=ax)
        ax.set_title(MODEL_LABELS[model])
        ax.axis("off")

    save_figure(filename)
    plt.show()

plot_example_graphs()

## 2. Feature Engineering and Static Graph Analysis

For each node we compute:

- degree
- PageRank
- clustering coefficient
- betweenness centrality
- closeness centrality

We then compare the models using repeated graph realizations.

In [ ]:
def compute_pagerank(G: nx.Graph, alpha: float = 0.85) -> dict:
    return nx.pagerank(G, alpha=alpha)

def extract_node_features(G: nx.Graph, model_name: str) -> pd.DataFrame:
    pagerank = compute_pagerank(G)
    degree_dict = dict(G.degree())
    clustering_dict = nx.clustering(G)
    betweenness_dict = nx.betweenness_centrality(G)
    closeness_dict = nx.closeness_centrality(G)

    df = pd.DataFrame({
        "node": list(G.nodes()),
        "degree": [degree_dict[node] for node in G.nodes()],
        "pagerank": [pagerank[node] for node in G.nodes()],
        "clustering_coef": [clustering_dict[node] for node in G.nodes()],
        "betweenness": [betweenness_dict[node] for node in G.nodes()],
        "closeness": [closeness_dict[node] for node in G.nodes()],
        "model": model_name,
    })

    df["log_degree"] = np.log1p(df["degree"])
    df["log_pagerank"] = np.log1p(df["pagerank"])
    return df

FEATURE_COLS = [
    "degree",
    "pagerank",
    "clustering_coef",
    "betweenness",
    "closeness",
    "log_degree",
    "log_pagerank",
]

def gini(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    if np.any(x < 0):
        raise ValueError("Gini coefficient is not defined for negative values.")
    if np.allclose(x.sum(), 0):
        return 0.0
    x = np.sort(x)
    n = len(x)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * x) / (n * np.sum(x))) - (n + 1) / n

In [ ]:
def plot_pagerank_distributions(example_seed: int = EXAMPLE_SEED, filename: str = "pagerank_distributions") -> None:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)

    for ax, model in zip(axes, MODEL_ORDER):
        G = generate_graph(model, n=N_ANALYSIS, seed=example_seed, **ANALYSIS_PARAMS[model])
        df = extract_node_features(G, model)
        ax.hist(df["pagerank"], bins=30)
        ax.set_title(MODEL_LABELS[model])
        ax.set_xlabel("PageRank")
        ax.grid(axis="y", alpha=0.25)

    axes[0].set_ylabel("Node count")
    save_figure(filename)
    plt.show()

plot_pagerank_distributions()

In [ ]:
def run_pca(df: pd.DataFrame, cols: list[str], n_components: int = 2):
    X = df[cols].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_scaled)

    out = df.copy()
    for i in range(n_components):
        out[f"PC{i + 1}"] = X_pca[:, i]
    return out, pca, scaler

def fit_best_gmm(df: pd.DataFrame, cols: list[str], k_range=range(2, 7), random_state: int = RANDOM_STATE):
    X = df[cols].values
    best_model = None
    best_k = None
    best_bic = np.inf

    for k in k_range:
        gmm = GaussianMixture(n_components=k, random_state=random_state)
        gmm.fit(X)
        bic = gmm.bic(X)
        if bic < best_bic:
            best_bic = bic
            best_k = k
            best_model = gmm

    out = df.copy()
    out["cluster"] = best_model.predict(X)
    return out, best_model, best_k, best_bic

In [ ]:
def analyze_graph(model: str, n: int, seed: int, params: dict) -> tuple[pd.DataFrame, dict]:
    G = generate_graph(model, n=n, seed=seed, **params)
    df = extract_node_features(G, model)

    df_pca, pca, _ = run_pca(df, FEATURE_COLS, n_components=2)
    df_clustered, _, best_k, _ = fit_best_gmm(df_pca, ["PC1", "PC2"], random_state=seed)

    result = {
        "model": model,
        "graph_seed": seed,
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "avg_degree": np.mean([d for _, d in G.degree()]),
        "pagerank_std": df["pagerank"].std(),
        "pagerank_max": df["pagerank"].max(),
        "pagerank_gini": gini(df["pagerank"].values),
        "degree_std": df["degree"].std(),
        "degree_pagerank_corr": df[["degree", "pagerank"]].corr().iloc[0, 1],
        "pca_var_pc1": pca.explained_variance_ratio_[0],
        "pca_var_pc2": pca.explained_variance_ratio_[1],
        "num_clusters": best_k,
    }
    return df_clustered, result

In [ ]:
all_node_results = []
all_graph_results = []

for graph_seed in range(NUM_GRAPH_SEEDS):
    for model in MODEL_ORDER:
        df_clustered, result = analyze_graph(
            model=model,
            n=N_ANALYSIS,
            seed=graph_seed,
            params=ANALYSIS_PARAMS[model],
        )
        all_node_results.append(df_clustered)
        all_graph_results.append(result)

nodes_df = pd.concat(all_node_results, ignore_index=True)
results_df = pd.DataFrame(all_graph_results)

results_df.head()

In [ ]:
static_summary = (
    results_df
    .groupby("model")[[
        "pagerank_std",
        "pagerank_max",
        "pagerank_gini",
        "degree_std",
        "degree_pagerank_corr",
        "num_clusters",
    ]]
    .agg(["mean", "std"])
    .reindex(MODEL_ORDER)
)
static_summary

In [ ]:
def summarize_metric_with_ci(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    rows = []
    for model in MODEL_ORDER:
        vals = df[df["model"] == model][metric].dropna()
        mean = vals.mean()
        std = vals.std(ddof=1)
        n = len(vals)
        se = std / np.sqrt(n)
        rows.append({
            "model": model,
            "mean": mean,
            "ci_lower": mean - 1.96 * se,
            "ci_upper": mean + 1.96 * se,
        })
    return pd.DataFrame(rows)

def plot_model_metric_ci(results_df: pd.DataFrame, metric: str, title: str, ylabel: str, filename: str) -> None:
    summary = summarize_metric_with_ci(results_df, metric)
    x = np.arange(len(summary))

    means = summary["mean"].to_numpy()
    lower = means - summary["ci_lower"].to_numpy()
    upper = summary["ci_upper"].to_numpy() - means

    fig, ax = plt.subplots(figsize=(6.6, 4.6))
    ax.errorbar(x, means, yerr=np.array([lower, upper]), fmt="o", capsize=5)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[m] for m in summary["model"]])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    save_figure(filename)
    plt.show()

plot_model_metric_ci(
    results_df,
    metric="pagerank_gini",
    title="PageRank inequality by graph model",
    ylabel="PageRank Gini coefficient",
    filename="pagerank_gini_by_model",
)

plot_model_metric_ci(
    results_df,
    metric="degree_std",
    title="Degree variability by graph model",
    ylabel="Standard deviation of degree",
    filename="degree_std_by_model",
)

plot_model_metric_ci(
    results_df,
    metric="num_clusters",
    title="Selected GMM clusters by graph model",
    ylabel="Number of selected clusters",
    filename="num_clusters_by_model",
)

## 3. PCA and GMM Visualization

In [ ]:
combined_features = nodes_df[FEATURE_COLS].values
combined_scaled = StandardScaler().fit_transform(combined_features)

combined_pca = PCA(n_components=2, random_state=RANDOM_STATE)
combined_pca_values = combined_pca.fit_transform(combined_scaled)

nodes_df = nodes_df.copy()
nodes_df["global_PC1"] = combined_pca_values[:, 0]
nodes_df["global_PC2"] = combined_pca_values[:, 1]

combined_pca.explained_variance_ratio_

In [ ]:
def plot_global_pca_clean(nodes_df: pd.DataFrame, filename: str = "combined_pca_projection") -> None:
    fig, ax = plt.subplots(figsize=(7.2, 5.5))
    for model in MODEL_ORDER:
        subset = nodes_df[nodes_df["model"] == model]
        ax.scatter(
            subset["global_PC1"],
            subset["global_PC2"],
            alpha=0.25,
            s=18,
            label=MODEL_LABELS[model],
        )
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title("Combined PCA projection of node features")
    ax.legend(frameon=False)
    ax.grid(alpha=0.20)
    save_figure(filename)
    plt.show()

plot_global_pca_clean(nodes_df)

In [ ]:
def plot_clustered_pca_panel(nodes_df: pd.DataFrame, example_seed: int = EXAMPLE_SEED, filename: str = "clustered_pca_panels") -> None:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharex=False, sharey=False)
    for ax, model in zip(axes, MODEL_ORDER):
        subset = nodes_df[(nodes_df["model"] == model) & (nodes_df["graph_seed"] == example_seed)]
        for cluster in sorted(subset["cluster"].unique()):
            cluster_df = subset[subset["cluster"] == cluster]
            ax.scatter(cluster_df["PC1"], cluster_df["PC2"], s=24, alpha=0.6, label=f"Cluster {cluster}")
        ax.set_title(MODEL_LABELS[model])
        ax.set_xlabel("PC1")
        ax.grid(alpha=0.2)
    axes[0].set_ylabel("PC2")
    axes[-1].legend(frameon=False, loc="best")
    save_figure(filename)
    plt.show()

plot_clustered_pca_panel(nodes_df)

## 4. Simulated Spread and Antidote Intervention

We use a simple **SI** process:

- each node is either susceptible or infected
- infected nodes remain infected
- infected nodes infect susceptible neighbors with probability `beta`

To model **antidote allocation**, we compare three strategies before spread begins:

1. **No intervention**
2. **Targeted intervention**: remove the top 5 PageRank nodes
3. **Random intervention**: remove 5 random nodes

Spread starts from a **random remaining node**, so the comparison isolates the effect of the intervention strategy.

In [ ]:
def simulate_si_spread(G: nx.Graph, initial_infected, beta: float = 0.20, steps: int = 20, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    infected = {initial_infected}
    infected_counts = [1]
    infected_fraction = [1 / G.number_of_nodes()]

    for _ in range(steps):
        new_infected = set(infected)
        for node in infected:
            for neighbor in G.neighbors(node):
                if neighbor not in infected and rng.random() < beta:
                    new_infected.add(neighbor)

        infected = new_infected
        infected_counts.append(len(infected))
        infected_fraction.append(len(infected) / G.number_of_nodes())

    return infected_counts, infected_fraction

def time_to_threshold(fractions, threshold: float):
    for t, frac in enumerate(fractions):
        if frac >= threshold:
            return t
    return np.nan

def area_under_curve(y):
    y = np.asarray(y, dtype=float)
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, dx=1)
    return np.trapz(y, dx=1)

def remove_nodes_by_strategy(G: nx.Graph, df: pd.DataFrame, strategy: str, k: int = 5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    G_mod = G.copy()

    if strategy == "none":
        removed_nodes = []
    elif strategy == "targeted_pagerank":
        removed_nodes = (
            df.sort_values("pagerank", ascending=False)
              .head(k)["node"]
              .tolist()
        )
        G_mod.remove_nodes_from(removed_nodes)
    elif strategy == "random":
        removed_nodes = rng.choice(list(G.nodes()), size=k, replace=False).tolist()
        G_mod.remove_nodes_from(removed_nodes)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    return G_mod, removed_nodes

In [ ]:
def run_intervention_spread_experiment(
    model: str,
    n: int,
    graph_params: dict,
    graph_seed: int,
    intervention_strategy: str,
    beta: float = SPREAD_BETA,
    steps: int = SPREAD_STEPS,
    k_remove: int = K_REMOVE,
    sim_reps: int = SPREAD_REPS,
) -> pd.DataFrame:
    rng_graph = np.random.default_rng(graph_seed)

    G = generate_graph(model, n=n, seed=graph_seed, **graph_params)
    df = extract_node_features(G, model)

    G_mod, removed_nodes = remove_nodes_by_strategy(
        G,
        df,
        strategy=intervention_strategy,
        k=k_remove,
        rng=rng_graph,
    )

    if G_mod.number_of_nodes() == 0:
        return pd.DataFrame()

    remaining_nodes = list(G_mod.nodes())
    records = []

    for sim_rep in range(sim_reps):
        rng = np.random.default_rng(100_000 * graph_seed + sim_rep)
        initial_infected = rng.choice(remaining_nodes)

        counts, fractions = simulate_si_spread(
            G_mod,
            initial_infected=initial_infected,
            beta=beta,
            steps=steps,
            rng=rng,
        )

        records.append({
            "model": model,
            "graph_seed": graph_seed,
            "strategy": intervention_strategy,
            "sim_rep": sim_rep,
            "nodes_remaining": G_mod.number_of_nodes(),
            "edges_remaining": G_mod.number_of_edges(),
            "final_fraction_infected": fractions[-1],
            "t_25": time_to_threshold(fractions, 0.25),
            "t_50": time_to_threshold(fractions, 0.50),
            "t_75": time_to_threshold(fractions, 0.75),
            "auc_infected": area_under_curve(fractions),
            "curve": fractions,
        })

    return pd.DataFrame(records)

In [ ]:
intervention_results = []

for graph_seed in range(NUM_GRAPH_SEEDS):
    for model in MODEL_ORDER:
        for strategy in STRATEGY_ORDER:
            intervention_results.append(
                run_intervention_spread_experiment(
                    model=model,
                    n=N_ANALYSIS,
                    graph_params=ANALYSIS_PARAMS[model],
                    graph_seed=graph_seed,
                    intervention_strategy=strategy,
                    beta=SPREAD_BETA,
                    steps=SPREAD_STEPS,
                    k_remove=K_REMOVE,
                    sim_reps=SPREAD_REPS,
                )
            )

intervention_df = pd.concat(intervention_results, ignore_index=True)
intervention_df.head()

In [ ]:
def mean_ci(series, z: float = 1.96) -> pd.Series:
    x = pd.Series(series).dropna()
    n = len(x)
    if n == 0:
        return pd.Series({
            "mean": np.nan,
            "std": np.nan,
            "n": 0,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
        })
    mean = x.mean()
    std = x.std(ddof=1) if n > 1 else 0.0
    se = std / np.sqrt(n) if n > 1 else 0.0
    return pd.Series({
        "mean": mean,
        "std": std,
        "n": n,
        "ci_lower": mean - z * se,
        "ci_upper": mean + z * se,
    })

summary_rows = []
for metric in ["final_fraction_infected", "t_25", "t_50", "t_75", "auc_infected"]:
    for (model, strategy), group in intervention_df.groupby(["model", "strategy"]):
        stats = mean_ci(group[metric])
        summary_rows.append({
            "model": model,
            "strategy": strategy,
            "metric": metric,
            "mean": stats["mean"],
            "std": stats["std"],
            "n": stats["n"],
            "ci_lower": stats["ci_lower"],
            "ci_upper": stats["ci_upper"],
        })

intervention_summary_df = pd.DataFrame(summary_rows)
intervention_summary_df.head()

In [ ]:
def plot_metric_with_ci_clean(summary_df: pd.DataFrame, metric: str, title: str, ylabel: str, filename: str) -> None:
    subset = summary_df[summary_df["metric"] == metric].copy()

    fig, ax = plt.subplots(figsize=(9, 5))
    width = 0.22
    x = np.arange(len(MODEL_ORDER))

    for j, strategy in enumerate(STRATEGY_ORDER):
        means = []
        lower_err = []
        upper_err = []

        for model in MODEL_ORDER:
            row = subset[(subset["model"] == model) & (subset["strategy"] == strategy)].iloc[0]
            means.append(row["mean"])
            lower_err.append(row["mean"] - row["ci_lower"])
            upper_err.append(row["ci_upper"] - row["mean"])

        offset = (j - 1) * width
        ax.bar(x + offset, means, width=width, label=STRATEGY_LABELS[strategy])
        ax.errorbar(
            x + offset,
            means,
            yerr=np.array([lower_err, upper_err]),
            fmt="none",
            capsize=4,
        )

    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.25)
    save_figure(filename)
    plt.show()

plot_metric_with_ci_clean(
    intervention_summary_df,
    metric="final_fraction_infected",
    title="Final outbreak size by graph model and intervention strategy",
    ylabel="Final fraction infected",
    filename="final_fraction_infected_ci",
)

plot_metric_with_ci_clean(
    intervention_summary_df,
    metric="t_50",
    title="Time to 50% infection by graph model and intervention strategy",
    ylabel="Time steps to 50% infection",
    filename="t50_ci",
)

plot_metric_with_ci_clean(
    intervention_summary_df,
    metric="auc_infected",
    title="Cumulative infection burden by graph model and intervention strategy",
    ylabel="Area under infection curve",
    filename="auc_ci",
)

## 5. Paired Comparisons by Graph Realization

To make the intervention comparison stronger, we compare strategies **within the same graph realization**.

For each `(model, graph_seed)` pair, we average spread outcomes across simulation replicates and then compute:

- targeted minus random
- targeted minus none

For final infected fraction and infection burden, **more negative values are better** because they indicate that targeted removal reduced spread more.

In [ ]:
graph_level = (
    intervention_df
    .groupby(["model", "graph_seed", "strategy"])
    .agg({
        "final_fraction_infected": "mean",
        "t_50": "mean",
        "auc_infected": "mean",
    })
    .reset_index()
)

pivot = graph_level.pivot_table(
    index=["model", "graph_seed"],
    columns="strategy",
    values=["final_fraction_infected", "t_50", "auc_infected"],
)

paired_results = []
for (model, graph_seed), row in pivot.iterrows():
    paired_results.append({
        "model": model,
        "graph_seed": graph_seed,
        "delta_final_targeted_vs_random":
            row["final_fraction_infected"]["targeted_pagerank"]
            - row["final_fraction_infected"]["random"],
        "delta_final_targeted_vs_none":
            row["final_fraction_infected"]["targeted_pagerank"]
            - row["final_fraction_infected"]["none"],
        "delta_t50_targeted_vs_random":
            row["t_50"]["targeted_pagerank"]
            - row["t_50"]["random"],
        "delta_auc_targeted_vs_random":
            row["auc_infected"]["targeted_pagerank"]
            - row["auc_infected"]["random"],
    })

paired_df = pd.DataFrame(paired_results)
paired_df.head()

In [ ]:
paired_summary = (
    paired_df
    .groupby("model")[[
        "delta_final_targeted_vs_random",
        "delta_final_targeted_vs_none",
        "delta_t50_targeted_vs_random",
        "delta_auc_targeted_vs_random",
    ]]
    .agg(["mean", "std"])
    .reindex(MODEL_ORDER)
)
paired_summary

In [ ]:
def plot_paired_difference_boxplot(paired_df: pd.DataFrame, column: str, title: str, ylabel: str, filename: str) -> None:
    fig, ax = plt.subplots(figsize=(7.6, 5))
    data = [paired_df[paired_df["model"] == model][column].dropna() for model in MODEL_ORDER]
    ax.boxplot(data, labels=[MODEL_LABELS[m] for m in MODEL_ORDER], widths=0.6)
    ax.axhline(0, linestyle="--", linewidth=1.5)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.25)
    save_figure(filename)
    plt.show()

plot_paired_difference_boxplot(
    paired_df,
    column="delta_final_targeted_vs_random",
    title="Paired effect of targeted vs random intervention",
    ylabel="Difference in final infected fraction\n(targeted - random)",
    filename="paired_targeted_vs_random_final_fraction",
)

## 6. Mean Spread Curves under Intervention

The final figure below shows mean spread curves with percentile bands for each intervention strategy, separated by graph model.

In [ ]:
def plot_mean_spread_curves_by_model(intervention_df: pd.DataFrame, filename: str = "mean_spread_curves_by_model") -> None:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)

    for ax, model in zip(axes, MODEL_ORDER):
        model_df = intervention_df[intervention_df["model"] == model]

        for strategy in STRATEGY_ORDER:
            subset = model_df[model_df["strategy"] == strategy]
            curves = np.vstack(subset["curve"].to_numpy())
            mean_curve = curves.mean(axis=0)
            lower = np.percentile(curves, 2.5, axis=0)
            upper = np.percentile(curves, 97.5, axis=0)
            t = np.arange(curves.shape[1])

            ax.plot(t, mean_curve, label=STRATEGY_LABELS[strategy])
            ax.fill_between(t, lower, upper, alpha=0.18)

        ax.set_title(MODEL_LABELS[model])
        ax.set_xlabel("Time step")
        ax.grid(alpha=0.25)

    axes[0].set_ylabel("Fraction infected")
    axes[-1].legend(frameon=False, loc="lower right")
    save_figure(filename)
    plt.show()

plot_mean_spread_curves_by_model(intervention_df)

## 7. Exported Figures

All figures were saved to `figures/` as both `.pdf` and `.png`. The code below prints the filenames so they are easy to reference from your TeX report.

In [ ]:
exported_files = sorted([p.name for p in FIG_DIR.iterdir() if p.is_file()])
pd.DataFrame({"exported_figure_file": exported_files})